In [ ]:
import duckdb

In [ ]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [ ]:
con.execute("SELECT * FROM bronze_z0019")

In [ ]:
df = con.execute("SELECT * FROM bronze_z0019").fetchdf()
df.head(10)

In [ ]:
df = con.execute("""
                 SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                 FROM bronze_z0019
                 WHERE data_ingestao >= '2026-04-08'
                    """).fetchdf()
df.head(10)

In [ ]:
df = con.execute("""
                 SELECT *
                 FROM (
                 SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row
                 FROM bronze_z0019
                 WHERE data_ingestao >= '2026-04-08'
                ) WHERE row = 1
                    """).fetchdf()
df.head(10)

In [ ]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final.head(10)

In [ ]:
df_final = df_final.rename(columns={'NATBR': 'ID'})
df_final = df_final.rename(columns={'MAKTX': 'NOME'})
df_final = df_final.rename(columns={'WERKS': 'CATEGORIA'})
df_final = df_final.rename(columns={'MAINS': 'FORNECEDOR'})
df_final = df_final.rename(columns={'LABST': 'PREÇO'})
df_final.head(10)


In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        'ID': int,
        'NOME': str,
        'CATEGORIA': str,
        'FORNECEDOR': int,
        'PREÇO': float
        }
    )
df2.dtypes
#df2.head(10)

In [ ]:
con.execute("""
            CREATE TABLE IF NOT EXISTS produtos (
                ID BIGINT,
                NOME TEXT,
                CATEGORIA TEXT,
                FORNECEDOR BIGINT,
                PREÇO FLOAT
            )
        """)

In [ ]:
df2.head(10)

In [ ]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [ ]:
df_result = con.execute("SELECT * FROM produtos").fetchdf()
df_result.head(10)

In [ ]:
# con.close()

In [ ]:
con.execute("SHOW TABLES").fetchall()